In [1]:
!pip install pyDatalog

In [2]:
from pyDatalog import pyDatalog

pyDatalog.clear()

In [4]:
pyDatalog.create_terms ("Human, Mortal, X") # any variable, predicate/function

#Facts
+Human ('john')
+Human ('bob')

#Add rules: if Human(X) -> Moral(x)
Mortal(X) <= Human (X)

#Query
result = Mortal(X)
print (result)

X   
----
bob 
john


In [5]:

pyDatalog.create_terms ("Student, HasID, CanEnterLab,X")


#Add facts 

+Student ("john")
+HasID ("john")
+Student("bob")

# Add Rules
# For all of x where student and hasid
CanEnterLab (X) <= (Student(X) & HasID (X))

result = CanEnterLab (X)
print (result)

X   
----
john


In [6]:
# STEP 2: Rule with AND, OR, and NOT

pyDatalog.clear()

# Add extra predicates for OR/NOT example
pyDatalog.create_terms('Student, HasID, VIP, Suspended, CanEnterLab, X')

# Facts
+Student('john')
+HasID('john')

+Student('bob')
+VIP('bob')
+Suspended('bob')   # bob is suspended, should be blocked

+Student('alice')
+VIP('alice')       # alice has no ID but is VIP

# (Student AND HasID AND NOT Suspended) OR (Student AND VIP AND NOT Suspended)
CanEnterLab(X) <= (Student(X) & HasID(X) & ~Suspended(X))
CanEnterLab(X) <= (Student(X) & VIP(X) & ~Suspended(X))

# for all X,
# if (Student(X) AND HasID(X) AND NOT Suspended(X)) 
# OR (
# Student(X) AND VIP(X) AND NOT Suspended(X)), 
# then CanEnterLab(X)
result = CanEnterLab(X)
print('CanEnterLab(X):', result)

CanEnterLab(X): X    
-----
alice
john 


In [7]:
# STEP 3: Existential-style check

# Goal: check whether there exists at least one x such that CanEnterLab(x)

# 1) Run query (uses rules/facts from previous cell)
q = CanEnterLab(X)
print('Query result:', q)

# 2) Interpret existential
exists_someone = len(q.data) > 0
print('Exists someone who can enter:', exists_someone)

# Expected: True in current setup

Query result: X    
-----
alice
john 
Exists someone who can enter: True


In [8]:
# 1) Reset knowledge base
pyDatalog.clear()

# 2) Declare terms
pyDatalog.create_terms('Robot, BatteryPct, ObstacleDist, BatteryLow, ObstacleNear, NeedsRecharge, CanMove, R, B, D')

# 3) Simulate one sensor snapshot (Scenario A: risky)
robot = 'robot1'
battery_percent = 15
obstacle_distance = 0.3

# 4) Add raw numeric observation facts
+Robot(robot)
+BatteryPct(robot, battery_percent)
+ObstacleDist(robot, obstacle_distance)

# 5) Convert numeric facts to symbolic predicates via rules
BatteryLow(R) <= (BatteryPct(R, B) & (B < 20))
ObstacleNear(R) <= (ObstacleDist(R, D) & (D < 0.5))

# 6) Add task-level decision rules
NeedsRecharge(R) <= BatteryLow(R)
CanMove(R) <= (Robot(R) & ~BatteryLow(R) & ~ObstacleNear(R))

# 7) Query and interpret (Scenario A)
print('Scenario A raw values -> battery:', battery_percent, 'obstacle:', obstacle_distance)
print('BatteryLow:', list(BatteryLow(R)))
print('ObstacleNear:', list(ObstacleNear(R)))
print('NeedsRecharge:', list(NeedsRecharge(R)))
print('CanMove:', list(CanMove(R)))

# Expected for Scenario A:
# BatteryLow -> robot1
# ObstacleNear -> robot1
# NeedsRecharge -> robot1
# CanMove -> empty (blocked by low battery and obstacle)

Scenario A raw values -> battery: 15 obstacle: 0.3
BatteryLow: [('robot1',)]
ObstacleNear: [('robot1',)]
NeedsRecharge: [('robot1',)]
CanMove: []


In [9]:
pyDatalog.clear()
pyDatalog.create_terms('Robot, BatteryPct, ObstacleDist, BatteryLow, ObstacleNear, NeedsRecharge, CanMove, R, B, D')

robot = 'robot1'
battery_percent = 80
obstacle_distance = 1.2

+Robot(robot)
+BatteryPct(robot, battery_percent)
+ObstacleDist(robot, obstacle_distance)

BatteryLow(R) <= (BatteryPct(R, B) & (B < 20))
ObstacleNear(R) <= (ObstacleDist(R, D) & (D < 0.5))
NeedsRecharge(R) <= BatteryLow(R)
CanMove(R) <= (Robot(R) & ~BatteryLow(R) & ~ObstacleNear(R))

print('\nScenario B raw values -> battery:', battery_percent, 'obstacle:', obstacle_distance)
print('BatteryLow:', list(BatteryLow(R)))
print('ObstacleNear:', list(ObstacleNear(R)))
print('NeedsRecharge:', list(NeedsRecharge(R)))
print('CanMove:', list(CanMove(R)))

# Expected for Scenario B:
# BatteryLow -> empty
# ObstacleNear -> empty
# NeedsRecharge -> empty
# CanMove -> robot1


Scenario B raw values -> battery: 80 obstacle: 1.2
BatteryLow: []
ObstacleNear: []
NeedsRecharge: []
CanMove: [('robot1',)]
